In [ ]:
!gdown 14lX_JgofYZLbIkjPf7rXmBOpVIfMudDo -O kaggle.json
!mkdir -p /root/.kaggle
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download thedevastator/dailydialog-multi-turn-dialog-with-intention-and
!unzip dailydialog-multi-turn-dialog-with-intention-and.zip -d .

Downloading...
From: https://drive.google.com/uc?id=14lX_JgofYZLbIkjPf7rXmBOpVIfMudDo
To: /content/kaggle.json
100% 69.0/69.0 [00:00<00:00, 253kB/s]
Dataset URL: https://www.kaggle.com/datasets/thedevastator/dailydialog-multi-turn-dialog-with-intention-and
License(s): CC0-1.0
Archive:  dailydialog-multi-turn-dialog-with-intention-and.zip
  inflating: ./test.csv              
  inflating: ./train.csv             
  inflating: ./validation.csv        


In [ ]:
import pandas as pd
import ast
import re

# Function to fix act and emotion columns (adds missing commas and converts to list)
def fix_and_eval_list(s):
    if isinstance(s, list):
        return s
    s_fixed = re.sub(r"(\d)\s+(?=\d)", r"\1, ", s)
    return ast.literal_eval(s_fixed)

# Function to parse dialog and extract utterances (between quotes)
def parse_dialog(s):
    utterances = re.findall(r"'(.*?)'|\"(.*?)\"", s)
    return [u[0] if u[0] else u[1] for u in utterances]

def load_and_prepare_flat_df(path, tokenizer, max_length=64):
    # Load CSV
    df = pd.read_csv(path)

    # Parse columns
    df["dialog"] = df["dialog"].apply(parse_dialog)
    df["act"] = df["act"].apply(fix_and_eval_list)
    df["emotion"] = df["emotion"].apply(fix_and_eval_list)

    # Flatten
    rows = []
    for dialog_id, row in df.iterrows():
        for turn_id, (utt, emo) in enumerate(zip(row["dialog"], row["emotion"])):
            rows.append({
                "dialog_id": dialog_id,
                "turn_id": turn_id,
                "utterance": utt,
                "label": emo
            })
    df_flat = pd.DataFrame(rows)

    # Tokenize (lists of token IDs and masks)
    tokenized = tokenizer(
        df_flat["utterance"].tolist(),
        padding=True,
        truncation=True,
        max_length=max_length
    )
    df_flat["input_ids"] = tokenized["input_ids"]
    df_flat["attention_mask"] = tokenized["attention_mask"]

    return df_flat



In [ ]:
from transformers import RobertaTokenizer, RobertaModel
import torch

# Load RoBERTa base tokenizer and model (frozen)
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaModel.from_pretrained('roberta-base')
model.eval()  # Evaluation mode (disable dropout)

# If using GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

In [ ]:
df_train_flat = load_and_prepare_flat_df("train.csv", tokenizer)
df_val_flat = load_and_prepare_flat_df("validation.csv", tokenizer)
df_test_flat = load_and_prepare_flat_df("test.csv", tokenizer)

In [ ]:
df_train_flat.head()

,dialog_id,turn_id,utterance,label,input_ids,attention_mask
0,0,0,"Say , Jim , how about going for a few beers af...",0,"[0, 34673, 2156, 2488, 2156, 141, 59, 164, 13,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,0,1,You know that is tempting but is really not g...,0,"[0, 370, 216, 14, 16, 25057, 53, 16, 269, 45, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,0,2,What do you mean ? It will help us to relax .,0,"[0, 653, 109, 47, 1266, 17487, 85, 40, 244, 20...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,0,3,Do you really think so ? I don't . It will ju...,0,"[0, 1832, 47, 269, 206, 98, 17487, 38, 218, 75...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,0,4,I guess you are right.But what shall we do ? ...,0,"[0, 38, 4443, 47, 32, 235, 4, 1708, 99, 5658, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [ ]:
import torch
from tqdm import tqdm
import numpy as np

# Store embeddings and labels
embeddings = []
labels = []

# Loop over the dataframe
for idx, row in tqdm(df_train_flat.iterrows(), total=len(df_train_flat)):
    input_ids = torch.tensor([row["input_ids"]]).to(device)
    attention_mask = torch.tensor([row["attention_mask"]]).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # [1, seq_len, hidden_dim]

        # mean pooling: sum(hidden * mask) / sum(mask)
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()  # [1, seq_len, hidden_dim]
        masked_embeddings = last_hidden_state * mask
        summed = masked_embeddings.sum(dim=1)  # [1, hidden_dim]
        counts = mask.sum(dim=1)  # [1, hidden_dim] (broadcasted)
        mean_pooled = summed / counts.clamp(min=1e-9)  # avoid division by zero

        embeddings.append(mean_pooled.cpu().numpy().squeeze())
        labels.append(row["label"])

# Convert to numpy arrays
embeddings = np.array(embeddings)
labels = np.array(labels)

# Save to disk
np.save("train_embeddings.npy", embeddings)
np.save("train_labels.npy", labels)


100%|██████████| 87170/87170 [14:04<00:00, 103.21it/s]


In [ ]:
import torch
from tqdm import tqdm
import numpy as np

# Store embeddings and labels
val_embeddings = []
val_labels = []

# Loop over the dataframe
for idx, row in tqdm(df_val_flat.iterrows(), total=len(df_val_flat)):
    input_ids = torch.tensor([row["input_ids"]]).to(device)
    attention_mask = torch.tensor([row["attention_mask"]]).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # [1, seq_len, hidden_dim]

        # mean pooling con attention mask
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        masked_embeddings = last_hidden_state * mask
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1)
        mean_pooled = summed / counts.clamp(min=1e-9)

        val_embeddings.append(mean_pooled.cpu().numpy().squeeze())
        val_labels.append(row["label"])

# Convert to numpy arrays
val_embeddings = np.array(val_embeddings)
val_labels = np.array(val_labels)

# Save to disk
np.save("val_embeddings.npy", val_embeddings)
np.save("val_labels.npy", val_labels)


100%|██████████| 8069/8069 [01:17<00:00, 103.60it/s]


In [ ]:
import torch
from tqdm import tqdm
import numpy as np

# Store embeddings and labels
test_embeddings = []
test_labels = []

# Loop over the dataframe
for idx, row in tqdm(df_test_flat.iterrows(), total=len(df_test_flat)):
    input_ids = torch.tensor([row["input_ids"]]).to(device)
    attention_mask = torch.tensor([row["attention_mask"]]).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # [1, seq_len, hidden_dim]

        # mean pooling con attention mask
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        masked_embeddings = last_hidden_state * mask
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1)
        mean_pooled = summed / counts.clamp(min=1e-9)

        test_embeddings.append(mean_pooled.cpu().numpy().squeeze())
        test_labels.append(row["label"])

# Convert to numpy arrays
test_embeddings = np.array(test_embeddings)
test_labels = np.array(test_labels)

# Save to disk
np.save("test_embeddings.npy", test_embeddings)
np.save("test_labels.npy", test_labels)


100%|██████████| 7740/7740 [01:14<00:00, 103.34it/s]


In [ ]:
import zipfile

# Train set
with zipfile.ZipFile('train_embeddings_labels.zip', 'w') as zipf:
    zipf.write('train_embeddings.npy')
    zipf.write('train_labels.npy')

# Validation set
with zipfile.ZipFile('val_embeddings_labels.zip', 'w') as zipf:
    zipf.write('val_embeddings.npy')
    zipf.write('val_labels.npy')

# Test set
with zipfile.ZipFile('test_embeddings_labels.zip', 'w') as zipf:
    zipf.write('test_embeddings.npy')
    zipf.write('test_labels.npy')
